In [1]:
import torch
import math

In [2]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [3]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [4]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=True)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=True)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=True)

In [5]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

In [6]:
queries = inputs @ W_query
keys = inputs @ W_key
values = inputs @ W_value

In [7]:
queries

tensor([[0.2309, 1.0966],
        [0.4306, 1.4551],
        [0.4300, 1.4343],
        [0.2355, 0.7990],
        [0.2983, 0.6565],
        [0.2568, 1.0533]], grad_fn=<MmBackward0>)

In [8]:
attn_score_2 = query_2 @ keys.T

In [9]:
attn_score_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [10]:
attn_score = queries @ keys.T

In [11]:
attn_score

tensor([[0.9231, 1.3545, 1.3241, 0.7910, 0.4032, 1.1330],
        [1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
        [1.2544, 1.8284, 1.7877, 1.0654, 0.5508, 1.5238],
        [0.6973, 1.0167, 0.9941, 0.5925, 0.3061, 0.8475],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707, 0.7307],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]],
       grad_fn=<MmBackward0>)

In [12]:
# Scale the attention scores by sqrt(d_out) before softmax.
#
# WHY scale at all:
#   attn_score is a dot product of query and key vectors. Softmax is sensitive
#   to the magnitude of its inputs: if one score is much larger than the rest,
#   softmax saturates and puts almost all the weight on that single entry while
#   pushing the others toward 0. That makes the attention distribution overly
#   "peaky" and, worse, the gradients through softmax vanish in that saturated
#   region, so the model stops learning. We want the scores in a moderate range.
#
# WHY the magnitude grows with dimension:
#   Assume each component of q and k is independent with mean 0 and variance 1.
#   The score is q . k = sum over i of (q_i * k_i), a sum of d_out terms. Each
#   term has variance 1, and variance adds for independent terms, so
#       Var(q . k) = d_out   ->   std dev = sqrt(d_out).
#   As d_out grows, the raw scores grow with it and softmax saturates.
#
# WHY divide by sqrt(d_out) (not d_out):
#   Dividing a random variable by a constant c scales its variance by c^2.
#   We want to bring the variance from d_out back down to 1, so we need
#   c^2 * d_out = 1  ->  c = 1 / sqrt(d_out).
#   Dividing by d_out itself would over-correct and crush the variance to
#   1/d_out. So sqrt(d_out) is the exact factor that normalizes the score
#   variance back to 1. (This is the "scaled" in scaled dot-product attention.)
attn_score = attn_score / math.sqrt(d_out)
attn_weights = torch.softmax(attn_score, dim=1)

In [13]:
attn_weights = torch.softmax(attn_score, dim=1)

In [14]:
attn_weights

tensor([[0.1551, 0.2104, 0.2059, 0.1413, 0.1074, 0.1799],
        [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        [0.1503, 0.2256, 0.2192, 0.1315, 0.0914, 0.1819],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.1206, 0.1769],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.1752],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]],
       grad_fn=<SoftmaxBackward0>)

In [15]:
attn_weights @ values 

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [29]:
import torch.nn as nn


class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))
    
    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        attn_score = queries @ keys.T
        attn_weights = torch.softmax(
            attn_score / keys.shape[-1] ** 0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec

In [30]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
sa_v1.forward(inputs)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [33]:
import torch.nn as nn


class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    
    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_score = queries @ keys.T
        attn_weights = torch.softmax(
            attn_score / keys.shape[-1] ** 0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec

In [34]:
torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in, d_out)
sa_v2.forward(inputs)

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)